# K-Fold Cross Validation

## Introduction

K-Fold Cross Validation is a technique to evaluate model performance by dividing data into K equal subsets and training the model K times, each time using a different subset as the test set.

**Why use it?**
- Better generalization estimate than a single train-test split
- Uses all data for both training and validation
- Reduces impact of random train-test variations
- Essential for hyperparameter tuning and model selection

## How It Works

**Process:**
1. Split data into K equal subsets (folds)
2. For each fold i (1 to K):
   - Use fold i as the test set
   - Use remaining K-1 folds as training set
   - Train model and record performance
3. Average the K performance scores

**Example (K=5):**
- Fold 1: Train on folds 2-5, test on fold 1
- Fold 2: Train on folds 1,3-5, test on fold 2
- ... repeat for folds 3, 4, 5
- Final score = average of all 5 scores

**Formula:** $\text{CV Score} = \frac{1}{K} \sum_{i=1}^{K} M_i$ where $M_i$ is the score from fold i

## Key Points

- **K=5 or K=10** are most common defaults
- **Stratified K-Fold**: Use for classification to maintain class distribution in each fold
- **Data leakage**: Always preprocess and scale within CV loop, never before splitting

## Variations

**Stratified K-Fold:** For classification, ensures each fold has similar class proportions to the full dataset. Use this for imbalanced datasets.

**Leave-One-Out CV (LOOCV):** K=n (one sample per fold). Most computationally expensive but least biased. Only for very small datasets.

**Time Series CV:** For temporal data, training always precedes validation (no random shuffling) to prevent information leakage.

## Choosing K

**Start with K=5 or K=10**
- K=5: Fast, works for most cases
- K=10: Standard choice, best bias-variance tradeoff

**Adjust based on:**
- **Small dataset (<1000)**: Use K=5 or K=10
- **Large dataset (>10K)**: Use K=3 or K=5 for speed
- **Complex model**: Use K=10
- **Limited computation**: Use K=3
- **Imbalanced classes**: Use Stratified K-Fold

## Common Mistakes

1. **Data Leakage:** Scale/preprocess WITHIN CV loop, not before splitting
2. **Imbalanced Data:** Use Stratified K-Fold, not regular K-Fold
3. **Time Series:** Don't shuffle—use time-ordered splits (training before validation)
4. **Hyperparameter Tuning:** Use nested CV (inner loop for tuning, outer for evaluation)
5. **Non-Reproducible:** Always set `random_state` for reproducibility

## Summary

**K-Fold Cross Validation:**
- Provides more reliable performance estimates than a single train-test split
- Default validation method for most ML projects
- Best practices: Use K=5/10, always use Stratified K-Fold for classification, preprocess within the CV loop

In [57]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
import numpy as np

In [58]:
digits=load_digits()

In [59]:
X_Train,X_Test,Y_Train,Y_Test=train_test_split(digits.data,digits.target,test_size=0.3)

### getScore Function Explanation

The `getScore` function is a helper function used to train a machine learning model and evaluate its performance.

### Description

The `getScore` function is a simple utility used in machine learning workflows to train a given model and evaluate its performance. It takes a model along with training and testing datasets as input. First, it fits (trains) the model using the training data (`X_Train` and `Y_Train`). After training, it evaluates how well the model performs on unseen data by calculating the score using the test data (`X_Test` and `Y_Test`). The function then returns this score, which helps in understanding the accuracy or effectiveness of the model. This is especially useful for quickly comparing different models.

In [60]:
def getScore(model,X_Train,X_Test,Y_Train,Y_Test):
    model.fit(X_Train,Y_Train)
    return model.score(X_Test,Y_Test)

In [61]:
getScore(LogisticRegression(),X_Train,X_Test,Y_Train,Y_Test)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.9648148148148148

In [62]:
getScore(SVC(),X_Train,X_Test,Y_Train,Y_Test)

0.9907407407407407

In [63]:
getScore(RandomForestClassifier(),X_Train,X_Test,Y_Train,Y_Test)

0.9722222222222222

### Description

This example is used only for understanding how K-Fold cross-validation divides data. It does not use any real dataset, but instead shows how the indices are split internally. The data is divided into 3 parts (folds), and in each iteration, different indices are selected as training and testing sets. This helps visualize how the splitting process works step by step.

In [64]:
kf=KFold(n_splits=3)
for train_index,test_index in kf.split([1,2,3,4,5,6,7,8,9]):
    print(train_index,test_index)

[3 4 5 6 7 8] [0 1 2]
[0 1 2 6 7 8] [3 4 5]
[0 1 2 3 4 5] [6 7 8]


### Manual K-Fold Cross-Validation Description

This example demonstrates a **manual implementation of K-Fold cross-validation** to compare multiple machine learning models.

Instead of using built-in functions like `cross_val_score`, the process is done step by step. The dataset is divided into different folds using K-Fold. In each iteration, the data is split into training and testing sets using the generated indices.

For every split:
- The model is trained on the training data
- The model is tested on the testing data
- The obtained score is stored in a list

Three different models are evaluated:
- Logistic Regression  
- Support Vector Machine (SVM)  
- Random Forest Classifier  

Separate lists are maintained to store scores for each model across all folds. After completing all iterations, the scores of each model are printed.

This approach is called a **manual method** because it explicitly shows how cross-validation works internally, making it easier to understand how models are trained and evaluated on different splits of the data.

In [65]:
score_l=[]
score_svm=[]
score_rf=[]

for train_index,test_index in kf.split(digits.data):
    X_Train,X_Test,Y_Train,Y_Test=digits.data[train_index],digits.data[test_index],digits.target[train_index],digits.target[test_index]

    score_l.append(getScore(LogisticRegression(),X_Train,X_Test,Y_Train,Y_Test))
    score_svm.append(getScore(SVC(),X_Train,X_Test,Y_Train,Y_Test))
    score_rf.append(getScore(RandomForestClassifier(),X_Train,X_Test,Y_Train,Y_Test))
print(f"Logistic Regression:{score_l}")
print(f"Support Vector Machine:{score_svm}")
print(f"Random Forest Classifier:{score_rf}")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
   

Logistic Regression:[0.9232053422370617, 0.9415692821368948, 0.9148580968280468]
Support Vector Machine:[0.9666110183639399, 0.9816360601001669, 0.9549248747913188]
Random Forest Classifier:[0.9232053422370617, 0.9532554257095158, 0.9265442404006677]


### Cross-Validation Using Built-in Method

This example demonstrates the use of a **built-in function** to perform cross-validation instead of doing it manually.

Here, `cross_val_score` is used to automatically handle the entire K-Fold cross-validation process. It splits the dataset into multiple folds, trains the model on different training sets, and evaluates it on corresponding test sets.

For each model:
- The dataset is automatically divided into folds
- The model is trained and tested across all folds
- A list of scores is returned, where each value represents performance on one fold

Three models are evaluated:
- Logistic Regression  
- Support Vector Machine (SVM)  
- Random Forest Classifier  

This approach is more efficient and concise compared to the manual method, as it performs all the splitting, training, and evaluation internally with a single function call.

In [67]:
print(f"Logistic Regression:{cross_val_score(LogisticRegression(),digits.data,digits.target)}")
print(f"Support Vector Machine:{cross_val_score(SVC(),digits.data,digits.target)}")
print(f"Random Forest Classifier:{cross_val_score(RandomForestClassifier(),digits.data,digits.target)}")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
   

Logistic Regression:[0.92222222 0.86944444 0.94150418 0.93871866 0.89693593]
Support Vector Machine:[0.96111111 0.94444444 0.98328691 0.98885794 0.93871866]
Random Forest Classifier:[0.92222222 0.92222222 0.95543175 0.9637883  0.92200557]
